In [1]:
import requests 
import pandas as pd

In [2]:
url = "https://data.elexon.co.uk/bmrs/api/v1/reference/bmunits/all"
response = requests.get(url, timeout=30)
data = response.json()
pd.set_option("display.max_columns", None)

df = pd.DataFrame(data)


In [3]:
for col in df.columns:
    print(col)

nationalGridBmUnit
elexonBmUnit
eic
fuelType
leadPartyName
bmUnitType
fpnFlag
bmUnitName
leadPartyId
demandCapacity
generationCapacity
productionOrConsumptionFlag
transmissionLossFactor
workingDayCreditAssessmentImportCapability
nonWorkingDayCreditAssessmentImportCapability
workingDayCreditAssessmentExportCapability
nonWorkingDayCreditAssessmentExportCapability
creditQualifyingStatus
demandInProductionFlag
gspGroupId
gspGroupName
interconnectorId


In [59]:
df1 = df[[
    "nationalGridBmUnit",
    "elexonBmUnit",
    "bmUnitName",
    "fuelType",
    "generationCapacity",
    "gspGroupName",
    "bmUnitType"
]]
display(df1.head(5))

,nationalGridBmUnit,elexonBmUnit,bmUnitName,fuelType,generationCapacity,gspGroupName,bmUnitType
0,ABERU-1,E_ABERDARE,Aberdare Power Station,None,15.400,South Wales,E
1,ABRBO-1,T_ABRBO-1,ABRBO-1,WIND,99.000,None,T
2,ABRTW-1,E_ABRTW-1,Auchrobert Wind Farm,WIND,36.000,South Scotland,E
3,ACHLW-1,C__PSMAR001,C__PSMAR001-AAA-ACH-183,WIND,11.282,North Scotland,S
4,ACHRW-1,T_ACHRW-1,AChruach Wind Farm,WIND,42.600,None,T


In [60]:
df1["fuelType"].value_counts(dropna=False)

fuelType
None       2530
WIND        287
OTHER       118
CCGT         65
NPSHYD       36
OCGT         34
BIOMASS      19
PS           16
NUCLEAR      16
COAL         10
INTELEC       2
INTGRNL       2
INTVKL        2
INTNED        1
INTFR         1
INTEW         1
INTIFA2       1
INTIRL        1
INTNSL        1
INTNEM        1
Name: count, dtype: int64

In [61]:
df1["gspGroupName"].value_counts(dropna=False)

gspGroupName
None                          1947
North Western                  121
Southern                       108
East Midlands                   99
South Scotland                  92
North Scotland                  89
Eastern                         87
Midlands                        84
Yorkshire                       81
South Western                   71
South Wales                     70
South Eastern                   63
Merseyside North Wales          60
Northern                        53
London                          37
Eastern GSP Group               17
SOUTH of SCOTLAND GSP           15
Merseyside and North Wales      11
NORTH of SCOTLAND GSP           10
LE Distribution                  8
Yorkshire Electricity            8
_F - Northern                    7
NORTHERN                         6
Name: count, dtype: int64

In [62]:
s = df1[(df1["gspGroupName"].str.contains("scotland", case=False, na=False))]
s["fuelType"].value_counts()

fuelType
WIND       50
OTHER      22
NPSHYD      4
BIOMASS     2
Name: count, dtype: int64

In [63]:
s_wind = s[s["fuelType"]=="WIND"] # rows where location is scotland and fuel is wind
display(s_wind.head(10))
s_wind["gspGroupName"].value_counts()

,nationalGridBmUnit,elexonBmUnit,bmUnitName,fuelType,generationCapacity,gspGroupName,bmUnitType
2,ABRTW-1,E_ABRTW-1,Auchrobert Wind Farm,WIND,36.000,South Scotland,E
3,ACHLW-1,C__PSMAR001,C__PSMAR001-AAA-ACH-183,WIND,11.282,North Scotland,S
392,AG-PSMA01,2__PSMAR001,2__PSMAR001,WIND,50.000,NORTH of SCOTLAND GSP,S
407,AIRSW-1,E_AIRSW-1,Airies,WIND,38.000,South Scotland,E
420,ASHWW-1,E_ASHWW-1,Andershaw Wind Farm,WIND,35.000,South Scotland,E
421,ASLVW-1,E_ASLVW-1,Assel Valley Wind Farm,WIND,26.000,South Scotland,E
422,BABAW-1,E_BABAW-1,Baillie Wind Farm,WIND,52.500,North Scotland,E
444,BETHW-1,E_BETHW-1,BETHW-1,WIND,29.750,North Scotland,E
453,BLARW-1,E_BLARW-1,E_BLARW-1,WIND,35.000,North Scotland,E
471,BNWKW-1,E_BNWKW-1,Burn of Whilk Windfarm,WIND,22.500,North Scotland,E


gspGroupName
North Scotland           34
South Scotland           14
NORTH of SCOTLAND GSP     1
SOUTH of SCOTLAND GSP     1
Name: count, dtype: int64

In [64]:
s_other = s[s["fuelType"]=="OTHER"]   # rows where location is scotland but fuel other
# display(s_other) 
# checked. All OTHER fueltypes are BESS

In [65]:
missing_wind = df1[df1["fuelType"].isna() & df1["bmUnitName"].str.contains("wind", case=False, na=False)]
missing_wind
# windfarms which we could add but no capacity so I think it makes sense to not include them.

,nationalGridBmUnit,elexonBmUnit,bmUnitName,fuelType,generationCapacity,gspGroupName,bmUnitType
572,CRGHD-1,T_CRGHD-1,Carraig Gheal Wind Farm,None,0.000,None,T
834,GFLDW-1,E_GFLDW-1,Goole Fields Wind Farm,None,32.800,Yorkshire,E
871,GRIFWD-1,T_GRIFWD-1,Griffin Windfarm Demand BMU,None,0.000,None,T
2955,SWDNB-1,E_SWDNB-1,Swindon,None,19.000,Southern,E


In [66]:
df1[df1["gspGroupName"].isna()]["fuelType"].value_counts()

fuelType
WIND       235
OTHER       80
CCGT        57
NPSHYD      32
OCGT        18
NUCLEAR     16
PS          16
BIOMASS     14
COAL        10
INTELEC      2
INTGRNL      2
INTVKL       2
INTNED       1
INTFR        1
INTEW        1
INTIFA2      1
INTIRL       1
INTNSL       1
INTNEM       1
Name: count, dtype: int64

In [67]:
no_loc_wind = df1[(df1["gspGroupName"].isna()) & (df1["fuelType"]=="WIND")]
display(no_loc_wind.head(6))

,nationalGridBmUnit,elexonBmUnit,bmUnitName,fuelType,generationCapacity,gspGroupName,bmUnitType
1,ABRBO-1,T_ABRBO-1,ABRBO-1,WIND,99.000,None,T
4,ACHRW-1,T_ACHRW-1,AChruach Wind Farm,WIND,42.600,None,T
5,ACHYW-1,None,None,WIND,None,None,None
6,AFTOW-1,T_AFTOW-1,Afton Wind Farm,WIND,50.000,None,T
410,AKGLW-1,None,None,WIND,None,None,None
411,AKGLW-2,T_AKGLW-2,AIKENGALL II,WIND,63.840,None,T


In [68]:
no_loc_wind.shape

(235, 7)

In [69]:
s_wind.shape

(50, 7)

In [71]:
winds = [s_wind, no_loc_wind]
bmu_list = pd.concat(winds)
full_bmu_excel = result.to_excel("wind_bmu_v2.xlsx")

In [72]:
wind_bmunits_gina = pd.read_csv("wind_bmunits_gina.csv")
wind_bmunits_gina

,elexonBmUnit,nationalGridBmUnit,bmUnitName,leadPartyName,generationCapacity,gspGroupId,gspGroupName,bmUnitType
0,T_ABRBO-1,ABRBO-1,ABRBO-1,Aberdeen Offshore Wind Farm,99.000,NaN,NaN,T
1,E_ABRTW-1,ABRTW-1,Auchrobert Wind Farm,Npower Commercial Gas Limited,36.000,_N,South Scotland,E
2,C__PSMAR001,ACHLW-1,C__PSMAR001-AAA-ACH-183,Smartestenergy Limited,11.282,_P,North Scotland,S
3,T_ACHRW-1,ACHRW-1,AChruach Wind Farm,Mobius Renewables Gen. (GB2,42.600,NaN,NaN,T
4,NaN,ACHYW-1,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
314,T_WLNYO-4,WLNYO-4,WALNEY_4,Walney Extension Ltd,330.000,NaN,NaN,T
315,T_WLNYO-4,WLNYO-4,WALNEY_4,Walney Extension Ltd,330.000,NaN,NaN,T
316,T_WLNYW-1,WLNYW-1,WALNEY_1,Walney Offshore Windfarms Ltd,182.000,NaN,NaN,T
317,NaN,WTGRW-1,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
wind_bmunits_gina_add_fuel = wind_bmunits_gina.merge(
    df1[["nationalGridBmUnit", "fuelType"]],
    on="nationalGridBmUnit",
    how="left")
wind_bmunits_gina_add_fuel["fuelType"].value_counts()

fuelType
WIND       289
NPSHYD      14
OTHER        6
CCGT         4
BIOMASS      4
OCGT         2
INTVKL       1
INTGRNL      1
Name: count, dtype: int64